# IF3170 Artificial Intelligence | Tugas Besar 2

This notebook serves as a template for the assignment. Please create a copy of this notebook to complete your work. You can add more code blocks, markdown blocks, or new sections if needed.


Group Number: 06

Group Members:
- Richard Christian 13523024
- Kenneth Poenadi Name 13523040
- Ivan Wirawan 13523046
- Bob Kunanda 13523086
- M Zahran Ramadhan 13523104

## Import Libraries

In [6]:
%pip install pydantic ydata-profiling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import datetime
import pickle
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 67
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

Note: you may need to restart the kernel to use updated packages.


## Import Dataset

In [7]:
df = pd.read_csv("../data/train.csv")

# Exploratory Data Analysis (Optional)

Exploratory Data Analysis (EDA) is a crucial step in the data analysis process that involves examining and visualizing data sets to uncover patterns, trends, anomalies, and insights. It is the first step before applying more advanced statistical and machine learning techniques. EDA helps you to gain a deep understanding of the data you are working with, allowing you to make informed decisions and formulate hypotheses for further analysis.

In [8]:
profile = ProfileReport(df, title="Data Profiling Report", explorative=True)
profile.to_file("../data/profile_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# 1. Split Training Set and Validation Set

Splitting the training and validation set works as an early diagnostic towards the performance of the model we train. This is done before the preprocessing steps to **avoid data leakage inbetween the sets**. If you want to use k-fold cross-validation, split the data later and do the cleaning and preprocessing separately for each split.

Note: For training, you should use the data contained in the `train` folder given by the TA. The `test` data is only used for kaggle submission.

In [9]:
def stratified_train_test_split(X, y, ids, test_size=0.2, random_state=67):
    np.random.seed(random_state)
    
    # Gabungkan untuk memudahkan splitting
    data = pd.concat([ids, X, y], axis=1)
    
    train_indices = []
    val_indices = []
    
    # Split untuk setiap kelas secara terpisah
    for class_label in y.unique():
        class_data = data[data['Target'] == class_label]
        indices = class_data.index.tolist()
        np.random.shuffle(indices)
        split_point = int(len(indices) * (1 - test_size))
        
        train_indices.extend(indices[:split_point])
        val_indices.extend(indices[split_point:])
    
    np.random.shuffle(train_indices)
    np.random.shuffle(val_indices)
    
    # Split data
    train_data = data.loc[train_indices]
    val_data = data.loc[val_indices]
    
    # Pisahkan kembali
    X_train = train_data.drop(columns=['Student_ID', 'Target'])
    y_train = train_data['Target']
    ids_train = train_data['Student_ID']
    
    X_val = val_data.drop(columns=['Student_ID', 'Target'])
    y_val = val_data['Target']
    ids_val = val_data['Student_ID']
    
    return X_train, X_val, y_train, y_val, ids_train, ids_val

# Pisahkan features dan target
X_full = df.drop(columns=['Student_ID', 'Target'])
y_full = df['Target']
ids_full = df['Student_ID']

print(f"Total data: {len(df)}")
print(f"\nDistribusi Target:")
print(y_full.value_counts())

# Lakukan splitting
X_train, X_val, y_train, y_val, ids_train, ids_val = stratified_train_test_split(
    X_full, y_full, ids_full, test_size=0.2, random_state=RANDOM_STATE
)


Total data: 3096

Distribusi Target:
Target
Graduate    1546
Dropout      994
Enrolled     556
Name: count, dtype: int64


# 2. Data Cleaning and Preprocessing

This step is the first thing to be done once a Data Scientist have grasped a general knowledge of the data. Raw data is **seldom ready for training**, therefore steps need to be taken to clean and format the data for the Machine Learning model to interpret.

By performing data cleaning and preprocessing, you ensure that your dataset is ready for model training, leading to more accurate and reliable machine learning results. These steps are essential for transforming raw data into a format that machine learning algorithms can effectively learn from and make predictions.

We will give some common methods for you to try, but you only have to **at least implement one method for each process**. For each step that you will do, **please explain the reason why did you do that process. Write it in a markdown cell under the code cell you wrote.**

## A. Data Cleaning

**Data cleaning** is the crucial first step in preparing your dataset for machine learning. Raw data collected from various sources is often messy and may contain errors, missing values, and inconsistencies. Data cleaning involves the following steps:

1. **Handling Missing Data:** Identify and address missing values in the dataset. This can include imputing missing values, removing rows or columns with excessive missing data, or using more advanced techniques like interpolation.

2. **Dealing with Outliers:** Identify and handle outliers, which are data points significantly different from the rest of the dataset. Outliers can be removed or transformed to improve model performance.

3. **Data Validation:** Check for data integrity and consistency. Ensure that data types are correct, categorical variables have consistent labels, and numerical values fall within expected ranges.

4. **Removing Duplicates:** Identify and remove duplicate rows, as they can skew the model's training process and evaluation metrics.

5. **Feature Engineering**: Create new features or modify existing ones to extract relevant information. This step can involve scaling, normalizing, or encoding features for better model interpretability.

### I. Handling Missing Data

Missing data can adversely affect the performance and accuracy of machine learning models. There are several strategies to handle missing data in machine learning:

1. **Data Imputation:**

    a. **Mean, Median, or Mode Imputation:** For numerical features, you can replace missing values with the mean, median, or mode of the non-missing values in the same feature. This method is simple and often effective when data is missing at random.

    b. **Constant Value Imputation:** You can replace missing values with a predefined constant value (e.g., 0) if it makes sense for your dataset and problem.

    c. **Imputation Using Predictive Models:** More advanced techniques involve using predictive models to estimate missing values. For example, you can train a regression model to predict missing numerical values or a classification model to predict missing categorical values.

2. **Deletion of Missing Data:**

    a. **Listwise Deletion:** In cases where the amount of missing data is relatively small, you can simply remove rows with missing values from your dataset. However, this approach can lead to a loss of valuable information.

    b. **Column (Feature) Deletion:** If a feature has a large number of missing values and is not critical for your analysis, you can consider removing that feature altogether.

3. **Domain-Specific Strategies:**

    a. **Domain Knowledge:** In some cases, domain knowledge can guide the imputation process. For example, if you know that missing values are related to a specific condition, you can impute them accordingly.

4. **Imputation Libraries:**

    a. **Scikit-Learn:** Scikit-Learn provides a `SimpleImputer` class that can handle basic imputation strategies like mean, median, and mode imputation.

    b. **Fancyimpute:** Fancyimpute is a Python library that offers more advanced imputation techniques, including matrix factorization, k-nearest neighbors, and deep learning-based methods.

The choice of imputation method should be guided by the nature of your data, the amount of missing data, the problem you are trying to solve, and the assumptions you are willing to make.

In [10]:
categorical_cols = [
    'Marital status', 'Application mode', 'Application order', 'Course',
    'Daytime/evening attendance\t', 'Previous qualification', 'Gender', 
    'Nacionality', "Mother's qualification", "Father's qualification", 
    "Mother's occupation", "Father's occupation", 'Educational special needs', 
    'International', 'Debtor', 'Tuition fees up to date', 'Scholarship holder', 
    'Displaced'
]

categorical_cols = [col for col in categorical_cols if col in X_train.columns]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

print(f"\nKolom kategorikal: {len(categorical_cols)}")
print(f"Kolom numerikal: {len(numerical_cols)}")

# Konversi kolom kategorikal ke string
X_train[categorical_cols] = X_train[categorical_cols].astype(str)
X_val[categorical_cols] = X_val[categorical_cols].astype(str)
df.info()


Kolom kategorikal: 18
Kolom numerikal: 18
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3096 entries, 0 to 3095
Data columns (total 38 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Student_ID                                      3096 non-null   int64  
 1   Marital status                                  3096 non-null   int64  
 2   Application mode                                3096 non-null   int64  
 3   Application order                               3096 non-null   int64  
 4   Course                                          3096 non-null   int64  
 5   Daytime/evening attendance	                     3096 non-null   int64  
 6   Previous qualification                          3096 non-null   int64  
 7   Previous qualification (grade)                  3096 non-null   float64
 8   Nacionality                                     3096 non-null   int64  
 9 

### II. Dealing with Outliers

Outliers are data points that significantly differ from the majority of the data. They can be unusually high or low values that do not fit the pattern of the rest of the dataset. Outliers can significantly impact model performance, so it is important to handle them properly.

Some methods to handle outliers:
1. **Imputation**: Replace with mean, median, or a boundary value.
2. **Clipping**: Cap values to upper and lower limits.
3. **Transformation**: Use log, square root, or power transformations to reduce their influence.
4. **Model-Based**: Use algorithms robust to outliers (e.g., tree-based models, Huber regression).

In [11]:
def detect_outliers_iqr(data, column):
    """
    Mendeteksi outlier menggunakan metode IQR.
    IQR = Q3 - Q1
    Outlier: nilai < Q1 - 1.5*IQR atau > Q3 + 1.5*IQR
    """
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_indices = data[(data[column] < lower_bound) | (data[column] > upper_bound)].index
    
    return lower_bound, upper_bound, outlier_indices

X_train_clean = X_train.copy()
X_val_clean = X_val.copy()

outlier_info = {}
for col in numerical_cols:
    lower, upper, outlier_idx = detect_outliers_iqr(X_train_clean, col)
    n_outliers = len(outlier_idx)
    
    if n_outliers > 0:
        outlier_info[col] = {
            'count': n_outliers,
            'percentage': (n_outliers / len(X_train_clean)) * 100,
            'lower_bound': lower,
            'upper_bound': upper
        }
        
        # Clipping outliers
        X_train_clean[col] = X_train_clean[col].clip(lower, upper)
        X_val_clean[col] = X_val_clean[col].clip(lower, upper)

print(f"Outliers di-handle untuk {len(outlier_info)} kolom")

Outliers di-handle untuk 15 kolom


### III. Remove Duplicates
Handling duplicate values is crucial because they can compromise data integrity, leading to inaccurate analysis and insights. Duplicate entries can bias machine learning models, causing overfitting and reducing their ability to generalize to new data. They also inflate the dataset size unnecessarily, increasing computational costs and processing times. Additionally, duplicates can distort statistical measures and lead to inconsistencies, ultimately affecting the reliability of data-driven decisions and reporting. Ensuring data quality by removing duplicates is essential for accurate, efficient, and consistent analysis.

In [12]:
n_duplicates = X_train_clean.duplicated().sum()
if n_duplicates > 0:
    before_len = len(X_train_clean)
    train_with_target = pd.concat([X_train_clean, y_train], axis=1)
    train_with_target = train_with_target.drop_duplicates()
    
    X_train_clean = train_with_target.drop(columns=['Target'])
    y_train = train_with_target['Target']
    
    after_len = len(X_train_clean)
    print(f"Removed {before_len - after_len} duplicates")
else:
    print("Tidak ada duplicates")

Tidak ada duplicates



### IV. Feature Engineering

**Feature engineering** involves creating new features (input variables) or transforming existing ones to improve the performance of machine learning models. Feature engineering aims to enhance the model's ability to learn patterns and make accurate predictions from the data. It's often said that "good features make good models."

1. **Feature Selection:** Feature engineering can involve selecting the most relevant and informative features from the dataset. Removing irrelevant or redundant features not only simplifies the model but also reduces the risk of overfitting.

2. **Creating New Features:** Sometimes, the existing features may not capture the underlying patterns effectively. In such cases, engineers create new features that provide additional information. For example:
   
   - **Polynomial Features:** Engineers may create new features by taking the square, cube, or other higher-order terms of existing numerical features. This can help capture nonlinear relationships.
   
   - **Interaction Features:** Interaction features are created by combining two or more existing features. For example, if you have features "length" and "width," you can create an "area" feature by multiplying them.

3. **Binning or Discretization:** Continuous numerical features can be divided into bins or categories. For instance, age values can be grouped into bins like "child," "adult," and "senior."

4. **Domain-Specific Feature Engineering:** Depending on the domain and problem, engineers may create domain-specific features. For example, in fraud detection, features related to transaction history and user behavior may be engineered to identify anomalies.

Feature engineering is both a creative and iterative process. It requires a deep understanding of the data, domain knowledge, and experimentation to determine which features will enhance the model's predictive power.

In [13]:
X_train_fe = X_train_clean.copy()
X_val_fe = X_val_clean.copy()

# Total units approved
if all(col in X_train_fe.columns for col in ['Curricular units 1st sem (approved)', 
                                               'Curricular units 2nd sem (approved)']):
    X_train_fe['Total_Units_Approved'] = (
        X_train_fe['Curricular units 1st sem (approved)'] + 
        X_train_fe['Curricular units 2nd sem (approved)']
    )
    X_val_fe['Total_Units_Approved'] = (
        X_val_fe['Curricular units 1st sem (approved)'] + 
        X_val_fe['Curricular units 2nd sem (approved)']
    )
    numerical_cols.append('Total_Units_Approved')
    print("Feature: Total_Units_Approved")

# Average grade
if all(col in X_train_fe.columns for col in ['Curricular units 1st sem (grade)', 
                                               'Curricular units 2nd sem (grade)']):
    X_train_fe['Avg_Grade'] = (
        X_train_fe['Curricular units 1st sem (grade)'] + 
        X_train_fe['Curricular units 2nd sem (grade)']
    ) / 2
    X_val_fe['Avg_Grade'] = (
        X_val_fe['Curricular units 1st sem (grade)'] + 
        X_val_fe['Curricular units 2nd sem (grade)']
    ) / 2
    numerical_cols.append('Avg_Grade')
    print("Feature: Avg_Grade")

Feature: Total_Units_Approved
Feature: Avg_Grade


## B. Data Preprocessing

**Data preprocessing** is a broader step that encompasses both data cleaning and additional transformations to make the data suitable for machine learning algorithms. Its primary goals are:

1. **Feature Scaling:** Ensure that numerical features have similar scales. Common techniques include Min-Max scaling (scaling to a specific range) or standardization (mean-centered, unit variance).

2. **Encoding Categorical Variables:** Machine learning models typically work with numerical data, so categorical variables need to be encoded. This can be done using one-hot encoding, label encoding, or more advanced methods like target encoding.

3. **Handling Imbalanced Classes:** If dealing with imbalanced classes in a binary classification task, apply techniques such as oversampling, undersampling, or using different evaluation metrics to address class imbalance.

4. **Dimensionality Reduction:** Reduce the number of features using techniques like Principal Component Analysis (PCA) or feature selection to simplify the model and potentially improve its performance.

5. **Normalization:** Normalize data to achieve a standard distribution. This is particularly important for algorithms that assume normally distributed data.

### Notes on Preprocessing processes

It is advised to create functions or classes that have the same/similar type of inputs and outputs, so you can add, remove, or swap the order of the processes easily. You can implement the functions or classes by yourself

or

use `sklearn` library. To create a new preprocessing component in `sklearn`, implement a corresponding class that includes:
1. Inheritance to `BaseEstimator` and `TransformerMixin`
2. The method `fit`
3. The method `transform`

In [14]:
class StandardScaler:
    """
    Standard Scaler dari scratch.
    Formula: X_scaled = (X - mean) / std
    """
    def __init__(self):
        self.mean_ = None
        self.std_ = None
    
    def fit(self, X):
        """Hitung mean dan std dari training data"""
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        # Hindari pembagian dengan 0
        self.std_[self.std_ == 0] = 1
        return self
    
    def transform(self, X):
        """Transform data menggunakan mean dan std yang sudah di-fit"""
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Scaler belum di-fit!")
        return (X - self.mean_) / self.std_
    
    def fit_transform(self, X):
        """Fit dan transform sekaligus"""
        return self.fit(X).transform(X)

### I. Feature Scaling

**Feature scaling** is a preprocessing technique used in machine learning to standardize the range of independent variables or features of data. The primary goal of feature scaling is to ensure that all features contribute equally to the training process and that machine learning algorithms can work effectively with the data.

Here are the main reasons why feature scaling is important:

1. **Algorithm Sensitivity:** Many machine learning algorithms are sensitive to the scale of input features. If the scales of features are significantly different, some algorithms may perform poorly or take much longer to converge.

2. **Distance-Based Algorithms:** Algorithms that rely on distances or similarities between data points, such as k-nearest neighbors (KNN) and support vector machines (SVM), can be influenced by feature scales. Features with larger scales may dominate the distance calculations.

3. **Regularization:** Regularization techniques, like L1 (Lasso) and L2 (Ridge) regularization, add penalty terms based on feature coefficients. Scaling ensures that all features are treated equally in the regularization process.

Common methods for feature scaling include:

1. **Min-Max Scaling (Normalization):** This method scales features to a specific range, typically [0, 1]. It's done using the following formula:

   $$X' = \frac{X - X_{min}}{X_{max} - X_{min}}$$

   - Here, $X$ is the original feature value, $X_{min}$ is the minimum value of the feature, and $X_{max}$ is the maximum value of the feature.  
<br />
<br />
2. **Standardization (Z-score Scaling):** This method scales features to have a mean (average) of 0 and a standard deviation of 1. It's done using the following formula:

   $$X' = \frac{X - \mu}{\sigma}$$

   - $X$ is the original feature value, $\mu$ is the mean of the feature, and $\sigma$ is the standard deviation of the feature.  
<br />
<br />
3. **Robust Scaling:** Robust scaling is a method that scales features to the interquartile range (IQR) and is less affected by outliers. It's calculated as:

   $$X' = \frac{X - Q1}{Q3 - Q1}$$

   - $X$ is the original feature value, $Q1$ is the first quartile (25th percentile), and $Q3$ is the third quartile (75th percentile) of the feature.  
<br />
<br />
4. **Log Transformation:** In cases where data is highly skewed or has a heavy-tailed distribution, taking the logarithm of the feature values can help stabilize the variance and improve scaling.

The choice of scaling method depends on the characteristics of your data and the requirements of your machine learning algorithm. **Min-max scaling and standardization are the most commonly used techniques and work well for many datasets.**

Scaling should be applied separately to each training and test set to prevent data leakage from the test set into the training set. Additionally, **some algorithms may not require feature scaling, particularly tree-based models.**

In [15]:
scaler = StandardScaler()
X_train_scaled = X_train_fe.copy()
X_val_scaled = X_val_fe.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_fe[numerical_cols])
X_val_scaled[numerical_cols] = scaler.transform(X_val_fe[numerical_cols])
print(" Scaling selesai")

 Scaling selesai


### II. Feature Encoding

**Feature encoding**, also known as **categorical encoding**, is the process of converting categorical data (non-numeric data) into a numerical format so that it can be used as input for machine learning algorithms. Most machine learning models require numerical data for training and prediction, so feature encoding is a critical step in data preprocessing.

Categorical data can take various forms, including:

1. **Nominal Data:** Categories with no intrinsic order, like colors or country names.  

2. **Ordinal Data:** Categories with a meaningful order but not necessarily equidistant, like education levels (e.g., "high school," "bachelor's," "master's").

There are several common methods for encoding categorical data:

1. **Label Encoding:**

   - Label encoding assigns a unique integer to each category in a feature.
   - It's suitable for ordinal data where there's a clear order among categories.
   - For example, if you have an "education" feature with values "high school," "bachelor's," and "master's," you can encode them as 0, 1, and 2, respectively.
<br />
<br />
2. **One-Hot Encoding:**

   - One-hot encoding creates a binary (0 or 1) column for each category in a nominal feature.
   - It's suitable for nominal data where there's no inherent order among categories.
   - Each category becomes a new feature, and the presence (1) or absence (0) of a category is indicated for each row.
<br />
<br />
3. **Target Encoding (Mean Encoding):**

   - Target encoding replaces each category with the mean of the target variable for that category.
   - It's often used for classification problems.

In [16]:
class LabelEncoder:
    """
    Label Encoder dari scratch.
    Mengubah categorical values menjadi integer.
    """
    def __init__(self):
        self.label_mapping_ = {}
        self.inverse_mapping_ = {}
    
    def fit(self, X, columns):
        """Buat mapping dari categorical values ke integer"""
        for col in columns:
            unique_values = X[col].unique()
            mapping = {val: idx for idx, val in enumerate(unique_values)}
            self.label_mapping_[col] = mapping
            self.inverse_mapping_[col] = {idx: val for val, idx in mapping.items()}
        return self
    
    def transform(self, X, columns):
        """Transform categorical values ke integer"""
        X_encoded = X.copy()
        for col in columns:
            if col in self.label_mapping_:
                X_encoded[col] = X[col].map(self.label_mapping_[col])
                # Handle unseen values
                X_encoded[col] = X_encoded[col].fillna(-1).astype(int)
        return X_encoded
    
    def fit_transform(self, X, columns):
        """Fit dan transform sekaligus"""
        return self.fit(X, columns).transform(X, columns)


print("\n--- Feature Encoding ---")
label_encoder = LabelEncoder()
X_train_encoded = label_encoder.fit_transform(X_train_scaled, categorical_cols)
X_val_encoded = label_encoder.transform(X_val_scaled, categorical_cols)
print("Encoding selesai")


# Encode target
target_encoder = LabelEncoder()
y_train_encoded = pd.Series(
    target_encoder.fit_transform(pd.DataFrame({'Target': y_train}), ['Target'])['Target'].values,
    index=y_train.index
)
y_val_encoded = pd.Series(
    target_encoder.transform(pd.DataFrame({'Target': y_val}), ['Target'])['Target'].values,
    index=y_val.index
)
print("Target encoding selesai")


--- Feature Encoding ---
Encoding selesai
Target encoding selesai


### III. Handling Imbalanced Dataset

**Handling imbalanced datasets** is important because imbalanced data can lead to several issues that negatively impact the performance and reliability of machine learning models. Here are some key reasons:

1. **Biased Model Performance**:

 - Models trained on imbalanced data tend to be biased towards the majority class, leading to poor performance on the minority class. This can result in misleading accuracy metrics.

2. **Misleading Accuracy**:

 - High overall accuracy can be misleading in imbalanced datasets. For example, if 95% of the data belongs to one class, a model that always predicts the majority class will have 95% accuracy but will fail to identify the minority class.

3. **Poor Generalization**:

 - Models trained on imbalanced data may not generalize well to new, unseen data, especially if the minority class is underrepresented.


Some methods to handle imbalanced datasets:
1. **Resampling Methods**:

 - Oversampling: Increase the number of instances in the minority class by duplicating or generating synthetic samples (e.g., SMOTE).
 - Undersampling: Reduce the number of instances in the majority class to balance the dataset.

2. **Evaluation Metrics**:

 - Use appropriate evaluation metrics such as precision, recall, F1-score, ROC-AUC, and confusion matrix instead of accuracy to better assess model performance on imbalanced data.

3. **Algorithmic Approaches**:

 - Use algorithms that are designed to handle imbalanced data, such as decision trees, random forests, or ensemble methods.
 - Adjust class weights in algorithms to give more importance to the minority class.

In [17]:
# Write your code here

### IV. Data Normalization

Data normalization is used to achieve a standard distribution. Without normalization, models or processes that rely on the assumption of normality may not work correctly. Normalization helps reduce the magnitude effect and ensures numerical stability during optimization.

In [18]:
# Write your code here

### V. Dimensionality Reduction

Dimensionality reduction is a technique used in data preprocessing to reduce the number of input features (dimensions) in a dataset while retaining as much important information as possible. It is essential when dealing with high-dimensional data, where too many features can cause problems like increased computational costs, overfitting, and difficulty in visualization. Reducing dimensions simplifies the data, making it easier to analyze and improving the performance of machine learning models.

One of the main approaches to dimensionality reduction is feature extraction. Feature extraction creates new, smaller sets of features that capture the essence of the original data. Common techniques include:

1. **Principal Component Analysis (PCA)**: Converts correlated features into a smaller number of uncorrelated "principal components."
2. **t-SNE (t-Distributed Stochastic Neighbor Embedding)**: A visualization-focused method to project high-dimensional data into 2D or 3D spaces.
3. **Autoencoders**: Neural networks that learn compressed representations of the data.

In [19]:
# Write your code here

# 3. Compile Preprocessing Pipeline

All of the preprocessing classes or functions defined earlier will be compiled in this step.

If you use sklearn to create preprocessing classes, you can list your preprocessing classes in the Pipeline object sequentially, and then fit and transform your data.

In [20]:
# from sklearn.pipeline import Pipeline

# # Note: You can add or delete preprocessing components from this pipeline

# pipe = Pipeline([("imputer", FeatureImputer()),
#                  ("featurecreator", FeatureCreator()),
#                  ("scaler", FeatureScaler()),
#                  ("encoder", FeatureEncoder())])

# train_set = pipe.fit_transform(train_set)
# val_set = pipe.transform(val_set)

In [21]:
# # Your code should work up until this point
# train_set = pipe.fit_transform(train_set)
# val_set = pipe.transform(val_set)

or create your own here

In [22]:
# Write your code here

# 4. Modeling and Validation

Modelling is the process of building your own machine learning models to solve specific problems, or in this assignment context, predicting the target feature `attack_cat`. Validation is the process of evaluating your trained model using the validation set or cross-validation method and providing some metrics that can help you decide what to do in the next iteration of development.

## A. DTL

In [23]:
class DecisionTreeNode:
    """Node untuk Decision Tree"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # Untuk leaf node

class DecisionTreeClassifier:
    """
    Decision Tree menggunakan CART algorithm dengan Gini Impurity.
    Sesuai requirement: SALAH SATU dari ID3, C4.5, atau CART.
    """
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.tree = None
    
    def _gini_impurity(self, y):
        """
        Menghitung Gini Impurity.
        Gini = 1 - Σ(p_i²)
        """
        if len(y) == 0:
            return 0
        proportions = np.bincount(y) / len(y)
        return 1 - np.sum(proportions ** 2)
    
    def _split_data(self, X, y, feature_idx, threshold):
        """Split data berdasarkan feature dan threshold"""
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        return X[left_mask], X[right_mask], y[left_mask], y[right_mask]
    
    def _information_gain(self, y, y_left, y_right):
        """Hitung information gain dari split"""
        parent_gini = self._gini_impurity(y)
        n = len(y)
        n_left, n_right = len(y_left), len(y_right)
        
        if n_left == 0 or n_right == 0:
            return 0
        
        weighted_gini = (n_left / n) * self._gini_impurity(y_left) + \
                        (n_right / n) * self._gini_impurity(y_right)
        
        return parent_gini - weighted_gini
    
    def _best_split(self, X, y):
        """Cari best feature dan threshold untuk split"""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                X_left, X_right, y_left, y_right = self._split_data(X, y, feature_idx, threshold)
                
                if len(y_left) < self.min_samples_leaf or len(y_right) < self.min_samples_leaf:
                    continue
                
                gain = self._information_gain(y, y_left, y_right)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth=0):
        """Build tree secara rekursif"""
        n_samples = len(y)
        n_classes = len(np.unique(y))
        
        # Stopping criteria
        if depth >= self.max_depth or n_samples < self.min_samples_split or n_classes == 1:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Cari best split
        best_feature, best_threshold = self._best_split(X, y)
        
        if best_feature is None:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Split data dan build subtrees
        X_left, X_right, y_left, y_right = self._split_data(X, y, best_feature, best_threshold)
        left_subtree = self._build_tree(X_left, y_left, depth + 1)
        right_subtree = self._build_tree(X_right, y_right, depth + 1)
        
        return DecisionTreeNode(best_feature, best_threshold, left_subtree, right_subtree)
    
    def fit(self, X, y):
        """Train decision tree"""
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        
        self.tree = self._build_tree(X, y)
        return self
    
    def _predict_sample(self, x, node):
        """Prediksi untuk single sample"""
        if node.value is not None:
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Prediksi untuk multiple samples"""
        if isinstance(X, pd.DataFrame):
            X = X.values
        
        predictions = np.array([self._predict_sample(x, self.tree) for x in X])
        return predictions
    
    def save(self, filepath):
        """Save model ke file"""
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
    
    @staticmethod
    def load(filepath):
        """Load model dari file"""
        with open(filepath, 'rb') as f:
            return pickle.load(f)

print("\n✓ DecisionTreeClassifier (CART) berhasil dibuat")


✓ DecisionTreeClassifier (CART) berhasil dibuat


## B. Logistic Regression

In [24]:
class LogisticRegression:
    """
    Logistic Regression dari scratch menggunakan Gradient Descent.
    Support multiclass dengan One-vs-Rest strategy.
    """
    def __init__(self, learning_rate=0.01, n_iterations=1000, regularization='l2', lambda_param=0.01):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.regularization = regularization
        self.lambda_param = lambda_param
        self.weights = None
        self.bias = None
        self.classes = None
    
    def _sigmoid(self, z):
        """Sigmoid activation function"""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def _compute_loss(self, y, y_pred, weights):
        """Compute loss dengan regularization"""
        m = len(y)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        # Binary cross-entropy loss
        loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
        
        # Add regularization
        if self.regularization == 'l2':
            loss += (self.lambda_param / (2 * m)) * np.sum(weights ** 2)
        elif self.regularization == 'l1':
            loss += (self.lambda_param / m) * np.sum(np.abs(weights))
        
        return loss
    
    def _fit_binary(self, X, y):
        """Fit untuk binary classification"""
        m, n = X.shape
        
        # Initialize parameters
        weights = np.zeros(n)
        bias = 0
        
        # Gradient descent
        for i in range(self.n_iterations):
            # Forward pass
            linear_output = np.dot(X, weights) + bias
            y_pred = self._sigmoid(linear_output)
            
            # Compute gradients
            dw = (1 / m) * np.dot(X.T, (y_pred - y))
            db = (1 / m) * np.sum(y_pred - y)
            
            # Add regularization to gradients
            if self.regularization == 'l2':
                dw += (self.lambda_param / m) * weights
            elif self.regularization == 'l1':
                dw += (self.lambda_param / m) * np.sign(weights)
            
            # Update parameters
            weights -= self.learning_rate * dw
            bias -= self.learning_rate * db
        
        return weights, bias
    
    def fit(self, X, y):
        """Train logistic regression (One-vs-Rest untuk multiclass)"""
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        if n_classes == 2:
            # Binary classification
            y_binary = (y == self.classes[1]).astype(int)
            self.weights, self.bias = self._fit_binary(X, y_binary)
        else:
            # Multiclass: One-vs-Rest
            self.weights = []
            self.bias = []
            
            for class_label in self.classes:
                y_binary = (y == class_label).astype(int)
                w, b = self._fit_binary(X, y_binary)
                self.weights.append(w)
                self.bias.append(b)
            
            self.weights = np.array(self.weights)
            self.bias = np.array(self.bias)
        
        return self
    
    def predict_proba(self, X):
        """Prediksi probabilitas"""
        if isinstance(X, pd.DataFrame):
            X = X.values
        
        if len(self.classes) == 2:
            linear_output = np.dot(X, self.weights) + self.bias
            proba = self._sigmoid(linear_output)
            return np.vstack([1 - proba, proba]).T
        else:
            # Multiclass
            probas = []
            for i in range(len(self.classes)):
                linear_output = np.dot(X, self.weights[i]) + self.bias[i]
                proba = self._sigmoid(linear_output)
                probas.append(proba)
            
            probas = np.array(probas).T
            # Normalize probabilities
            probas = probas / probas.sum(axis=1, keepdims=True)
            return probas
    
    def predict(self, X):
        """Prediksi class"""
        probas = self.predict_proba(X)
        return self.classes[np.argmax(probas, axis=1)]
    
    def save(self, filepath):
        """Save model"""
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
    
    @staticmethod
    def load(filepath):
        """Load model"""
        with open(filepath, 'rb') as f:
            return pickle.load(f)

print("✓ LogisticRegression berhasil dibuat")

✓ LogisticRegression berhasil dibuat


## C. SVM

In [25]:
class SVM:
    """
    SVM dari scratch dengan ONE-AGAINST-ONE strategy untuk multiclass.
    Sesuai requirement dari foto.
    """
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param
        self.n_iterations = n_iterations
        self.classifiers = {}  # Dictionary untuk menyimpan binary classifiers
        self.classes = None
    
    def _compute_hinge_loss(self, y, y_pred, weights):
        """
        Compute hinge loss dengan regularization.
        Hinge loss = max(0, 1 - y * y_pred)
        """
        m = len(y)
        hinge_losses = np.maximum(0, 1 - y * y_pred)
        loss = np.mean(hinge_losses)
        
        # Add L2 regularization
        loss += (self.lambda_param / 2) * np.sum(weights ** 2)
        
        return loss
    
    def _fit_binary(self, X, y):
        """
        Fit binary SVM classifier.
        y harus berisi nilai -1 dan 1
        """
        m, n = X.shape
        
        # Initialize parameters
        weights = np.zeros(n)
        bias = 0
        
        # Gradient descent
        for iteration in range(self.n_iterations):
            # Compute linear output
            linear_output = np.dot(X, weights) + bias
            
            # Compute condition for hinge loss
            condition = y * linear_output < 1
            
            # Compute gradients
            dw = np.zeros_like(weights)
            db = 0
            
            # Hinge loss gradient
            dw = self.lambda_param * weights
            dw -= np.dot(X[condition].T, y[condition]) / m
            db = -np.sum(y[condition]) / m
            
            # Update parameters
            weights -= self.learning_rate * dw
            bias -= self.learning_rate * db
        
        return weights, bias
    
    def fit(self, X, y):
        """
        Train SVM dengan ONE-AGAINST-ONE strategy.
        Untuk n kelas, akan membuat n*(n-1)/2 binary classifiers.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        # Train binary classifier untuk setiap pasangan kelas
        for i in range(n_classes):
            for j in range(i + 1, n_classes):
                class_i = self.classes[i]
                class_j = self.classes[j]
                
                # Ambil data untuk 2 kelas ini saja
                mask = (y == class_i) | (y == class_j)
                X_binary = X[mask]
                y_binary = y[mask]
                
                # Convert ke -1 dan 1
                y_binary = np.where(y_binary == class_i, -1, 1)
                
                # Train binary classifier
                weights, bias = self._fit_binary(X_binary, y_binary)
                
                # Simpan classifier
                self.classifiers[(class_i, class_j)] = (weights, bias)
        
        return self
    
    def _decision_function(self, X, class_pair):
        """Compute decision function untuk class pair"""
        weights, bias = self.classifiers[class_pair]
        return np.dot(X, weights) + bias
    
    def predict(self, X):
        """
        Prediksi menggunakan voting dari semua binary classifiers.
        Setiap classifier vote untuk salah satu dari 2 kelasnya.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        
        n_samples = X.shape[0]
        votes = np.zeros((n_samples, len(self.classes)))
        
        # Dapatkan vote dari setiap binary classifier
        for (class_i, class_j), (weights, bias) in self.classifiers.items():
            decision_values = np.dot(X, weights) + bias
            
            # Vote untuk class_i jika decision_value < 0, else class_j
            class_i_idx = np.where(self.classes == class_i)[0][0]
            class_j_idx = np.where(self.classes == class_j)[0][0]
            
            for idx in range(n_samples):
                if decision_values[idx] < 0:
                    votes[idx, class_i_idx] += 1
                else:
                    votes[idx, class_j_idx] += 1
        
        # Prediksi adalah kelas dengan vote terbanyak
        predictions = self.classes[np.argmax(votes, axis=1)]
        return predictions
    
    def save(self, filepath):
        """Save model"""
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
    
    @staticmethod
    def load(filepath):
        """Load model"""
        with open(filepath, 'rb') as f:
            return pickle.load(f)

print("✓ SVM (One-Against-One) berhasil dibuat")

✓ SVM (One-Against-One) berhasil dibuat


## D. Improvements (Optional)

- **Visualize the model evaluation result**

This will help you to understand the details more clearly about your model's performance. From the visualization, you can see clearly if your model is leaning towards a class than the others. (Hint: confusion matrix, ROC-AUC curve, etc.)

- **Explore the hyperparameters of your models**

Each models have their own hyperparameters. And each of the hyperparameter have different effects on the model behaviour. You can optimize the model performance by finding the good set of hyperparameters through a process called **hyperparameter tuning**. (Hint: Grid search, random search, bayesian optimization)

- **Cross-validation**

Cross-validation is a critical technique in machine learning and data science for evaluating and validating the performance of predictive models. It provides a more **robust** and **reliable** evaluation method compared to a hold-out (single train-test set) validation. Though, it requires more time and computing power because of how cross-validation works. (Hint: k-fold cross-validation, stratified k-fold cross-validation, etc.)

In [26]:
def calculate_metrics(y_true, y_pred, model_name):
    """Hitung accuracy, precision, recall, F1-score untuk setiap kelas"""
    from collections import Counter
    
    # Accuracy
    accuracy = np.mean(y_true == y_pred)
    
    # Per-class metrics
    classes = np.unique(y_true)
    precision_per_class = []
    recall_per_class = []
    f1_per_class = []
    
    for cls in classes:
        # True Positives, False Positives, False Negatives
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        
        # Precision, Recall, F1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precision_per_class.append(precision)
        recall_per_class.append(recall)
        f1_per_class.append(f1)
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision per class: {[f'{p:.4f}' for p in precision_per_class]}")
    print(f"  Recall per class: {[f'{r:.4f}' for r in recall_per_class]}")
    print(f"  F1-score per class: {[f'{f:.4f}' for f in f1_per_class]}")
    
    return accuracy, precision_per_class, recall_per_class, f1_per_class

## E. Submission
To predict the test set target feature and submit the results to the kaggle competition platform, do the following:
1. Create a new pipeline instance identical to the first in Data Preprocessing
2. With the pipeline, apply `fit_transform` to the original training set before splitting, then only apply `transform` to the test set.
3. Retrain the model on the preprocessed training set
4. Predict the test set
5. Make sure the submission contains the `id` and `attack_cat` column.

In [27]:
print("\n--- Training Decision Tree (CART) ---")
dt_model = DecisionTreeClassifier(max_depth=10, min_samples_split=20, min_samples_leaf=5)
dt_model.fit(X_train_encoded, y_train_encoded)

dt_pred_train = dt_model.predict(X_train_encoded)
dt_pred_val = dt_model.predict(X_val_encoded)

dt_acc_train, _, _, _ = calculate_metrics(y_train_encoded.values, dt_pred_train, "DT Train")
dt_acc_val, _, _, _ = calculate_metrics(y_val_encoded.values, dt_pred_val, "DT Validation")


# --- TRAIN LOGISTIC REGRESSION ---
print("\n--- Training Logistic Regression ---")
lr_model = LogisticRegression(learning_rate=0.01, n_iterations=1000, regularization='l2', lambda_param=0.01)
lr_model.fit(X_train_encoded, y_train_encoded)

lr_pred_train = lr_model.predict(X_train_encoded)
lr_pred_val = lr_model.predict(X_val_encoded)

lr_acc_train, _, _, _ = calculate_metrics(y_train_encoded.values, lr_pred_train, "LR Train")
lr_acc_val, _, _, _ = calculate_metrics(y_val_encoded.values, lr_pred_val, "LR Validation")


# --- TRAIN SVM (ONE-AGAINST-ONE) ---
print("\n--- Training SVM (One-Against-One) ---")
svm_model = SVM(learning_rate=0.001, lambda_param=0.01, n_iterations=1000)
svm_model.fit(X_train_encoded, y_train_encoded)

svm_pred_train = svm_model.predict(X_train_encoded)
svm_pred_val = svm_model.predict(X_val_encoded)

svm_acc_train, _, _, _ = calculate_metrics(y_train_encoded.values, svm_pred_train, "SVM Train")
svm_acc_val, _, _, _ = calculate_metrics(y_val_encoded.values, svm_pred_val, "SVM Validation")


--- Training Decision Tree (CART) ---

DT Train:
  Accuracy: 0.8327
  Precision per class: ['0.8557', '0.8545', '0.6989']
  Recall per class: ['0.8277', '0.9361', '0.5541']
  F1-score per class: ['0.8414', '0.8934', '0.6181']

DT Validation:
  Accuracy: 0.7166
  Precision per class: ['0.7513', '0.7739', '0.4138']
  Recall per class: ['0.7136', '0.8613', '0.3214']
  F1-score per class: ['0.7320', '0.8153', '0.3618']

--- Training Logistic Regression ---

DT Train:
  Accuracy: 0.8327
  Precision per class: ['0.8557', '0.8545', '0.6989']
  Recall per class: ['0.8277', '0.9361', '0.5541']
  F1-score per class: ['0.8414', '0.8934', '0.6181']

DT Validation:
  Accuracy: 0.7166
  Precision per class: ['0.7513', '0.7739', '0.4138']
  Recall per class: ['0.7136', '0.8613', '0.3214']
  F1-score per class: ['0.7320', '0.8153', '0.3618']

--- Training Logistic Regression ---

LR Train:
  Accuracy: 0.7285
  Precision per class: ['0.7242', '0.7425', '0.5122']
  Recall per class: ['0.7862', '0.9191'

In [ ]:
test_df = pd.read_csv("../data/test.csv")
test_ids = test_df["Student_ID"]
X_test = test_df.drop(columns=["Student_ID"])

# Preprocessing test data (SAME STEPS AS TRAINING)
print("\nPreprocessing test data...")

# Convert categorical
X_test[categorical_cols] = X_test[categorical_cols].astype(str)

# Feature engineering
if all(col in X_test.columns for col in ['Curricular units 1st sem (approved)', 
                                          'Curricular units 2nd sem (approved)']):
    X_test['Total_Units_Approved'] = (
        X_test['Curricular units 1st sem (approved)'] + 
        X_test['Curricular units 2nd sem (approved)']
    )

if all(col in X_test.columns for col in ['Curricular units 1st sem (grade)', 
                                          'Curricular units 2nd sem (grade)']):
    X_test['Avg_Grade'] = (
        X_test['Curricular units 1st sem (grade)'] + 
        X_test['Curricular units 2nd sem (grade)']
    ) / 2

# Align columns
X_test = X_test[X_train_encoded.columns]

# Scaling
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Encoding
X_test = label_encoder.transform(X_test, categorical_cols)

# Predict menggunakan model terbaik (pilih berdasarkan validation accuracy)
best_model_name = max([
    ("Decision Tree", dt_acc_val),
    ("Logistic Regression", lr_acc_val),
    ("SVM", svm_acc_val)
], key=lambda x: x[1])[0]

print(f"\nMenggunakan model terbaik: {best_model_name}")

if best_model_name == "Decision Tree":
    y_test_pred_encoded = dt_model.predict(X_test)
elif best_model_name == "Logistic Regression":
    y_test_pred_encoded = lr_model.predict(X_test)
else:
    y_test_pred_encoded = svm_model.predict(X_test)

# Decode predictions
y_test_pred = pd.Series(
    target_encoder.inverse_mapping_['Target'][pred] 
    for pred in y_test_pred_encoded
)

# Create submission
submission = pd.DataFrame({
    "Student_ID": test_ids,
    "Target": y_test_pred
})

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = f"../submission/submission_{timestamp}.csv"
submission.to_csv(submission_path, index=False)

print(f"\nSubmission saved to: {submission_path}")
print(f"Total predictions: {len(submission)}")



Preprocessing test data...

Menggunakan model terbaik: Logistic Regression

✓ Submission saved to: ../submission/submission_20251123_140620.csv
✓ Total predictions: 1328


# 6. Error Analysis

Based on all the process you have done until the modeling and evaluation step, write an analysis to support each steps you have taken to solve this problem. Write the analysis using the markdown block. Some questions that may help you in writing the analysis:

- Does my model perform better in predicting one class than the other? If so, why is that?
- To each models I have tried, which performs the best and what could be the reason?
- Is it better for me to impute or drop the missing data? Why?
- Does feature scaling help improve my model performance?
- etc...

`Provide your analysis here`